<a href="https://colab.research.google.com/github/ErickJester/expo-escom/blob/main/00_conteo_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Herramientas de Dataset — ExpoEscom
Menú con utilidades sobre una carpeta de Drive (incluye "Compartido conmigo").

**Opciones del menú:**
1. **Contabilizar** — cuenta las imágenes `.jpg` de una carpeta (recursivo)
   y muestra cuántas faltan para llegar a 100k. Incluye una sub-opción para
   contar **todos los datasets sumados, excepto la carpeta «otra»** (muestra
   el desglose por clase y el total).
2. **Contabilizar por dentro** — eliges una carpeta contenedora (ej. «otra»)
   y decides: **DESGLOSAR** (cuenta cada subcarpeta por separado + total, útil
   cuando una clase vive anidada como «noise» dentro de «otra») o **TOTAL**
   (todo lo de adentro como un solo número). Todo vía Drive API, así ve las
   subcarpetas aunque Drive todavía no las muestre en la web.
3. **Copiar al padre** — copia las imágenes de una subcarpeta a la carpeta
   **PADRE** («la padre de todos», ej. «hora de aventura») **dejando el
   original** en la subcarpeta (crea duplicados; Drive ya no permite
   multi-parent). Si la subcarpeta tiene a su vez **más subcarpetas**, pregunta
   si recolectar **TODO el árbol aplanado** en la padre, para que nada se quede
   atrapado en sub-subcarpetas. Salta los nombres que ya existen en destino.
4. **Completar a 100k (FUSE)** — augmenta **solo el déficit** leyendo/escribiendo
   por el acceso directo montado. ⚠️ Falla con `Input/output error` en carpetas
   muy grandes (FUSE no aguanta decenas de miles de archivos). Usa la opción [5].
5. **Completar a 100k vía API** — igual que [4] pero usando la Drive API (sin
   FUSE): lista, descarga y sube por API, así soporta carpetas enormes. No
   necesita el acceso directo en MyDrive, usa `FOLDER_ID` directo.
6. **Extraer comprimidos** — descarga un `.rar`/`.zip` que vive dentro de una
   subcarpeta (busca en todo el árbol), lo descomprime en Colab y sube sus
   imágenes **tal cual** (mismo nombre y tamaño, sin estandarizar ni renombrar)
   a la carpeta **PADRE**. Salta nombres ya presentes (re-run safe). Para `.rar`
   instala `unrar` automáticamente; `.zip` se maneja con Python nativo.

**⚠️ Solo la opción 4 necesita un acceso directo en tu "Mi unidad":**
como `FOLDER_ID` es "Compartido conmigo" no se monta como ruta. Crea **una sola
vez** un acceso directo a esa carpeta en tu *Mi unidad* (en Drive: clic derecho →
Organizar → Añadir acceso directo → Mi unidad) y pon su nombre en
`NOMBRE_ACCESO_DIRECTO` (cell-2). Las opciones [1], [2], [3], [5] y [6] no lo necesitan.

**Cómo usar:** corre todas las celdas de setup (de arriba) una vez, luego
ejecuta la última celda (Menú) cuantas veces quieras.

In [ ]:
VERSION = '4.9.0'

print('═' * 50)
print('🛠️  Herramientas de Dataset — ExpoEscom')
print(f'v{VERSION}')
print('═' * 50)

!pip install tqdm -q

import logging
logging.getLogger('googleapiclient.http').setLevel(logging.ERROR)

from google.colab import auth
from googleapiclient.discovery import build
from tqdm.notebook import tqdm

auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Autenticado con Drive API')

In [ ]:
# ════ ÚNICA CONFIGURACIÓN NECESARIA ══════════════════════════

FOLDER_ID = '1YwNZMW67NYGMb_g-74kG5gb2u_m_3c1J'

EXTENSIONES_IMAGEN = {'.jpg'}
TARGET_IMAGENES    = 100_000
print(f'Folder ID : {FOLDER_ID}')
print(f'Extensión : solo .jpg  |  Meta: {TARGET_IMAGENES:,} imágenes')

# ════ PARALELISMO (Colab Pro+: ~12 vCPU, 83 GB RAM) ═══════════
# Importante: salvo la augmentación, casi todo aquí lo limita la CUOTA de la
# Drive API, no el hardware. Las LECTURAS (listar/contar) tienen cuota alta →
# escalan bien con muchos hilos. Las ESCRITURAS (copy/create) tienen cuota
# baja → pasar de ~10 hilos solo genera 429 que el backoff absorbe sin ganar
# velocidad real. Por eso los conteos suben a 12 y las copias se quedan en 10.

# Opción [1] — conteo paralelo de varios datasets (lecturas, escala lineal)
NUM_COUNT_WORKERS = 12

# Opción [2] — copia paralela files().copy() (escritura, techo = cuota API)
NUM_COPY_WORKERS = 10

# ════ CONFIG OPCIÓN [3] — AUGMENTACIÓN (completar a 100k) ═════
# FOLDER_ID es "Compartido conmigo" → NO se monta como ruta directa.
# Crea UNA SOLA VEZ un acceso directo a esa carpeta en tu "Mi unidad":
#   en Drive → clic derecho en la carpeta → Organizar → Añadir acceso
#   directo → Mi unidad. Pon abajo el NOMBRE del acceso directo.
NOMBRE_ACCESO_DIRECTO = 'dataset'                       # ← ajusta si lo nombraste distinto
MOUNT_ROOT   = f'/content/drive/MyDrive/{NOMBRE_ACCESO_DIRECTO}'

MAX_AUG      = 30_000   # tope de seguridad: no augmentar más que esto por clase
JPEG_QUALITY = 95
SEED         = 42
NUM_WORKERS         = 8   # augmentación opción [3] — genera + sube en paralelo
NUM_AUG_API_WORKERS = 12  # augmentación opción [4] vía API (descarga+CPU+subida)
print(f'Acceso directo (opción 3): {MOUNT_ROOT}')
print(f'Hilos: conteo {NUM_COUNT_WORKERS} · copia {NUM_COPY_WORKERS} · '
      f'aug[3] {NUM_WORKERS} · aug[4] {NUM_AUG_API_WORKERS}')

In [ ]:
# ════ HELPERS DE DRIVE API ════════════════════════════════════
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True,
             pageSize=1000)
_FOLDER_MIME = 'application/vnd.google-apps.folder'


def _es_imagen(f):
    return Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN


def listar_contenido(parent_id):
    subcarpetas, sueltas, token = [], 0, None
    while True:
        resp = service.files().list(
            q=f"'{parent_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        for f in resp.get('files', []):
            if f['mimeType'] == _FOLDER_MIME:
                subcarpetas.append({'id': f['id'], 'name': f['name']})
            elif _es_imagen(f):
                sueltas += 1
        token = resp.get('nextPageToken')
        if not token:
            break
    subcarpetas.sort(key=lambda c: c['name'].lower())
    return subcarpetas, sueltas


def _contar_recursivo(folder_id, svc, pbar=None):
    """Núcleo de conteo: recorre subcarpetas con el `svc` dado (cada hilo
    trae el suyo → thread-safe). Si pasas `pbar`, avanza por cada .jpg."""
    total, stack = 0, [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = svc.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif _es_imagen(f):
                    total += 1
                    if pbar is not None:
                        pbar.update(1)
            token = resp.get('nextPageToken')
            if not token:
                break
    return total


def contar_imagenes(folder_id, desc='Contando .jpg'):
    with tqdm(desc=desc, unit=' jpg', dynamic_ncols=True) as pbar:
        return _contar_recursivo(folder_id, service, pbar)


def _servicio_por_hilo():
    """Devuelve una factoría de `service` thread-local (httplib2 no es
    thread-safe → un cliente por hilo)."""
    _tl = threading.local()
    def _svc():
        if not hasattr(_tl, 's'):
            _tl.s = build('drive', 'v3')
        return _tl.s
    return _svc


def copiar_contenido(origen_id, destino_id):
    """Copia TODOS los hijos directos de origen_id hacia destino_id en paralelo.
    El original SE QUEDA en la subcarpeta. Salta nombres ya existentes en destino
    para que los re-runs no dupliquen lo ya copiado.
    Devuelve (copiados, saltados), ambos {imagenes, carpetas, otros}."""
    import time

    # 1) recolectar hijos directos de origen
    print('  Listando archivos a copiar…')
    items, token = [], None
    while True:
        resp = service.files().list(
            q=f"'{origen_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        items.extend(resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break

    # 2) nombres ya presentes en destino (para no re-copiar en re-runs)
    print('  Revisando qué ya existe en destino…')
    existentes, token = set(), None
    while True:
        resp = service.files().list(
            q=f"'{destino_id}' in parents and trashed=false",
            fields='nextPageToken, files(name)',
            pageToken=token, **_ARGS).execute()
        existentes.update(f['name'] for f in resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break

    def _contar(f, d):
        if f['mimeType'] == _FOLDER_MIME:
            d['carpetas'] += 1
        elif _es_imagen(f):
            d['imagenes'] += 1
        else:
            d['otros'] += 1

    # 3) separar los que hay que copiar de los que se saltan
    copiados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    saltados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    to_copy = []
    for f in items:
        if f['mimeType'] == _FOLDER_MIME or f['name'] in existentes:
            _contar(f, saltados)
        else:
            to_copy.append(f)

    # 4) copiar en paralelo — un service por hilo (httplib2 no es thread-safe)
    _svc = _servicio_por_hilo()

    def _copiar_uno(f):
        for intento in range(4):
            try:
                _svc().files().copy(
                    fileId=f['id'],
                    body={'parents': [destino_id], 'name': f['name']},
                    fields='id', supportsAllDrives=True).execute()
                return True, f, None
            except Exception as e:
                err = str(e)
                if '429' in err or 'Rate Limit' in err or 'userRateLimitExceeded' in err:
                    time.sleep(2 ** intento)
                else:
                    return False, f, err
        return False, f, 'Rate limit tras 4 intentos'

    print(f'  {len(to_copy):,} a copiar · {sum(saltados.values()):,} saltados · {NUM_COPY_WORKERS} hilos')
    with ThreadPoolExecutor(max_workers=NUM_COPY_WORKERS) as executor:
        with tqdm(total=len(to_copy), desc='Copiando', unit=' archivo',
                  dynamic_ncols=True) as pbar:
            for ok, f, err in executor.map(_copiar_uno, to_copy):
                if ok:
                    _contar(f, copiados)
                else:
                    print(f'   ⚠️ No se pudo copiar {f["name"]}: {err}')
                pbar.update(1)

    return copiados, saltados


def construir_indice(parent_id, incluir_raiz=True):
    raiz = service.files().get(
        fileId=parent_id, fields='name', supportsAllDrives=True).execute()
    subs, sueltas = listar_contenido(parent_id)
    indice = {}
    if incluir_raiz:
        indice[0] = {'name': f'(TODA la carpeta «{raiz["name"]}»)',
                     'id': parent_id}
    for i, c in enumerate(subs, 1):
        indice[i] = c
    return raiz['name'], indice, sueltas


def mostrar_indice(indice, titulo='ÍNDICE DE CARPETAS'):
    print('═' * 50)
    print(titulo)
    print('═' * 50)
    for num, c in indice.items():
        print(f'  [{num:>2}]  {c["name"]}')
    print('═' * 50)


def pedir_opcion(indice, pregunta):
    while True:
        sel = input(f'\n{pregunta} (número): ').strip()
        if sel.isdigit() and int(sel) in indice:
            return int(sel)
        print('   ⚠️  Número inválido, intenta de nuevo.')


print('✅ Helpers listos')

In [ ]:
# ════ PIPELINE DE AUGMENTACIÓN (opción 3) ════════════════════
# Portado TAL CUAL del notebook 09_augmentacion_100k. Solo se
# parametriza por clase/ruta; la lógica de transforms, hilos y
# guardado es idéntica (así funciona bien y no se toca).
import os, re, random, shutil, time, subprocess, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torchvision
import torchvision.transforms as T
from PIL import Image

_TV = tuple(int(x) for x in torchvision.__version__.split('.')[:2])
VALID_EXT = {'.jpg'}   # dataset 100% .jpg (consistente con la opción [1])


def _listar_drive(path, retries=3):
    """Lista .jpg con shell `find` (resiste FUSE mejor que os.scandir)."""
    for intento in range(retries):
        try:
            result = subprocess.run(
                f'find "{path}" -maxdepth 1 -type f -iname "*.jpg"',
                shell=True, capture_output=True, text=True, timeout=180)
            if result.returncode == 0:
                return [l.strip() for l in result.stdout.splitlines() if l.strip()]
            raise OSError(result.stderr.strip())
        except Exception as e:
            print(f'  ⚠️  Intento {intento+1}/{retries}: {e}')
            if intento < retries - 1:
                from google.colab import drive
                print('  🔄 Re-montando Drive...')
                drive.mount('/content/drive', force_remount=True)
    raise RuntimeError('❌ No se pudo listar la carpeta tras varios intentos.')


def _nuevo_transform():
    """Pipeline de augmentación (un compose por hilo, thread-local)."""
    try:
        _rot = (T.RandomRotation(degrees=12, fill=(210, 210, 210))
                if _TV >= (0, 9) else T.RandomRotation(degrees=12))
    except TypeError:
        _rot = T.RandomRotation(degrees=12)
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        _rot,
        T.RandomResizedCrop(224, scale=(0.88, 1.0), ratio=(0.95, 1.05)),
        T.ColorJitter(brightness=0.20, contrast=0.15, saturation=0.08, hue=0.03),
    ])


def augmentar_clase(clase, class_path):
    """Completa `clase` hasta TARGET_IMAGENES augmentando SOLO el déficit.
    class_path = ruta MONTADA de la carpeta de la clase."""
    class_path = str(class_path)

    # ── Diagnóstico: contar y calcular déficit ───────────────
    print('📋 Listando .jpg (puede tardar ~20s con 70K+ imgs)...')
    source_paths = _listar_drive(class_path)
    print(f'   {len(source_paths):,} .jpg encontrados')

    pat_num = re.compile(r'(\d+)$')
    indices = [int(m.group(1)) for p in source_paths
               if (m := pat_num.search(Path(p).stem))]

    count_existing = len(source_paths)
    max_index      = max(indices) if indices else 0
    next_index     = max_index + 1
    count_needed   = max(0, TARGET_IMAGENES - count_existing)

    SEP = '─' * 55
    print(SEP)
    print(f'  Clase               : {clase}')
    print(f'  .jpg en Drive       : {count_existing:>10,}')
    print(f'  Índice máximo       : {max_index:>10,}  (próximo: {next_index:,})')
    print(f'  TARGET              : {TARGET_IMAGENES:>10,}')
    print(f'  A generar           : {count_needed:>10,}')
    print(f'  Sin número en nombre: {count_existing - len(indices):>9,}  (se usan como fuente)')
    print(SEP)

    # ── Guardián ─────────────────────────────────────────────
    if count_needed == 0:
        print(f'🎉 «{clase}» ya completa — nada que hacer.')
        return
    if count_needed > MAX_AUG:
        print(f'🚫 DETENIDO — necesitas {count_needed:,} augmentaciones')
        print(f'   pero el límite es {MAX_AUG:,} (MAX_AUG en config).')
        print(f'   Con solo {count_existing:,} fuentes, augmentar tanto')
        print(f'   dañaría el dataset. Consigue más imágenes reales primero.')
        return

    ratio = count_needed / count_existing if count_existing else 0
    print(f'✅ {count_needed:,} ≤ {MAX_AUG:,} — dentro del límite seguro.')
    print(f'   Ratio: 1 original genera ~{ratio:.2f} copias en promedio.')

    # ── Generar (paralelo, en local) ─────────────────────────
    LOCAL_OUT = f'/content/aug_work/{clase}'
    os.makedirs(LOCAL_OUT, exist_ok=True)

    _thread_local = threading.local()
    def get_transform():
        if not hasattr(_thread_local, 't'):
            _thread_local.t = _nuevo_transform()
        return _thread_local.t

    random.seed(SEED)
    shuffled = source_paths.copy()
    random.shuffle(shuffled)

    jobs = []
    for i in range(count_needed):
        src = shuffled[i % len(shuffled)]
        dst = os.path.join(LOCAL_OUT, f'{clase}_{next_index + i:06d}.jpg')
        jobs.append((src, dst))
    first_new, last_new = next_index, next_index + count_needed - 1

    def process_one(args):
        src, dst = args
        try:
            img = Image.open(src).convert('RGB')
            get_transform()(img).save(dst, 'JPEG', quality=JPEG_QUALITY)
            return True
        except Exception:
            return False

    print(f'🚀 Augmentando con {NUM_WORKERS} hilos...')
    t_start = time.time()
    generated = errors = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        with tqdm(total=count_needed, unit='img', desc=f'Aug {clase}',
                  dynamic_ncols=True) as pbar:
            for ok in executor.map(process_one, jobs):
                generated += ok
                errors += (not ok)
                pbar.update(1)
    elapsed = time.time() - t_start
    rate = generated / elapsed if elapsed > 0 else 0
    print(f'✅ {generated:,} imgs en {elapsed/60:.1f} min  ({rate:.0f} img/s)')
    if errors:
        print(f'⚠️  {errors} archivos fallaron (se ignoraron)')

    # ── Subir solo los nuevos a Drive (paralelo) ─────────────
    pat_new = re.compile(rf'^{re.escape(clase)}_?(\d+)\.jpg$', re.IGNORECASE)
    to_upload = sorted(
        f for f in os.listdir(LOCAL_OUT)
        if (m := pat_new.match(f)) and first_new <= int(m.group(1)) <= last_new)
    print(f'⏫ Subiendo {len(to_upload):,} archivos a Drive ({NUM_WORKERS} hilos)...')
    t_up = time.time()

    def _subir(fname):
        shutil.copy2(os.path.join(LOCAL_OUT, fname),
                     os.path.join(class_path, fname))

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        with tqdm(total=len(to_upload), unit='img', desc='Drive',
                  dynamic_ncols=True) as pbar:
            for _ in executor.map(_subir, to_upload):
                pbar.update(1)
    print(f'✅ Subida OK en {(time.time()-t_up)/60:.1f} min → {class_path}')

    # ── Resumen final ────────────────────────────────────────
    final = sum(1 for f in os.scandir(class_path)
                if Path(f.name).suffix.lower() in VALID_EXT)
    print('=' * 55)
    print(f'  RESUMEN  —  {clase}')
    print('=' * 55)
    print(f'  Originales   : {count_existing:>10,}')
    print(f'  Generadas    : {generated:>10,}')
    print(f'  Total Drive  : {final:>10,} / {TARGET_IMAGENES:,}')
    if final >= TARGET_IMAGENES:
        print('  🎉 COMPLETA — lista para entrenamiento')
    else:
        print(f'  ⚠️  Faltan {TARGET_IMAGENES-final:,} — re-ejecuta')
    print('=' * 55)


print('✅ Pipeline de augmentación listo')

In [ ]:
# ════ PIPELINE DE AUGMENTACIÓN VÍA API (opción 4) ════════════
# Igual que augmentar_clase pero SIN FUSE: lista, descarga y sube con
# la Drive API. Para carpetas tan grandes que `find` sobre el mount da
# Input/output error (ej. pokemon con decenas de miles de .jpg).
import io as _io
from googleapiclient.http import MediaIoBaseUpload


def augmentar_clase_api(folder_id, clase):
    """Completa `clase` hasta TARGET_IMAGENES augmentando SOLO el déficit,
    usando la Drive API (no el mount). folder_id = id de la carpeta de la clase."""
    # ── Listar .jpg vía API (id + name) ──────────────────────
    print('📋 Listando .jpg vía API (paginado)…')
    archivos, token = [], None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name)',
            pageToken=token, **_ARGS).execute()
        archivos.extend(f for f in resp.get('files', []) if _es_imagen(f))
        token = resp.get('nextPageToken')
        if not token:
            break

    pat_num = re.compile(r'(\d+)$')
    indices = [int(m.group(1)) for f in archivos
               if (m := pat_num.search(Path(f['name']).stem))]
    count_existing = len(archivos)
    max_index      = max(indices) if indices else 0
    next_index     = max_index + 1
    count_needed   = max(0, TARGET_IMAGENES - count_existing)

    SEP = '─' * 55
    print(SEP)
    print(f'  Clase               : {clase}')
    print(f'  .jpg en Drive       : {count_existing:>10,}')
    print(f'  Índice máximo       : {max_index:>10,}  (próximo: {next_index:,})')
    print(f'  TARGET              : {TARGET_IMAGENES:>10,}')
    print(f'  A generar           : {count_needed:>10,}')
    print(SEP)

    if count_needed == 0:
        print(f'🎉 «{clase}» ya completa — nada que hacer.')
        return
    if count_needed > MAX_AUG:
        print(f'🚫 DETENIDO — necesitas {count_needed:,} augmentaciones')
        print(f'   pero el límite es {MAX_AUG:,} (MAX_AUG en config).')
        print('   Con tan pocas fuentes, augmentar tanto dañaría el dataset.')
        return
    if count_existing == 0:
        print('🚫 No hay imágenes fuente en la carpeta.')
        return

    ratio = count_needed / count_existing
    print(f'✅ {count_needed:,} ≤ {MAX_AUG:,} — dentro del límite seguro.')
    print(f'   Ratio: 1 original genera ~{ratio:.2f} copias en promedio.')

    # ── Trabajos: descargar fuente → augmentar → subir (paralelo) ──
    random.seed(SEED)
    shuffled = archivos.copy()
    random.shuffle(shuffled)
    jobs = [(shuffled[i % len(shuffled)], next_index + i)
            for i in range(count_needed)]

    _tl = threading.local()
    def _svc():
        if not hasattr(_tl, 's'):
            _tl.s = build('drive', 'v3')
        return _tl.s
    def _tf():
        if not hasattr(_tl, 't'):
            _tl.t = _nuevo_transform()
        return _tl.t

    def _trabajo(job):
        src, idx = job
        for intento in range(4):
            try:
                svc = _svc()
                data = svc.files().get_media(
                    fileId=src['id'], supportsAllDrives=True).execute()
                img = Image.open(_io.BytesIO(data)).convert('RGB')
                out = _io.BytesIO()
                _tf()(img).save(out, 'JPEG', quality=JPEG_QUALITY)
                out.seek(0)
                svc.files().create(
                    body={'name': f'{clase}_{idx:06d}.jpg',
                          'parents': [folder_id]},
                    media_body=MediaIoBaseUpload(out, mimetype='image/jpeg'),
                    fields='id', supportsAllDrives=True).execute()
                return True
            except Exception as e:
                err = str(e)
                if '429' in err or 'Rate Limit' in err or 'userRateLimitExceeded' in err:
                    time.sleep(2 ** intento)
                else:
                    return False
        return False

    print(f'🚀 Augmentando vía API con {NUM_AUG_API_WORKERS} hilos…')
    t0, generated = time.time(), 0
    with ThreadPoolExecutor(max_workers=NUM_AUG_API_WORKERS) as ex:
        with tqdm(total=count_needed, unit='img', desc=f'Aug {clase}',
                  dynamic_ncols=True) as pbar:
            for ok in ex.map(_trabajo, jobs):
                generated += ok
                pbar.update(1)
    elapsed = time.time() - t0
    errors  = count_needed - generated
    rate    = generated / elapsed if elapsed > 0 else 0
    print(f'✅ {generated:,} imgs en {elapsed/60:.1f} min  ({rate:.0f} img/s)')
    if errors:
        print(f'⚠️  {errors} fallaron (re-ejecuta para completar el resto)')

    final = count_existing + generated
    print('=' * 55)
    print(f'  RESUMEN  —  {clase}')
    print('=' * 55)
    print(f'  Originales   : {count_existing:>10,}')
    print(f'  Generadas    : {generated:>10,}')
    print(f'  Total Drive  : {final:>10,} / {TARGET_IMAGENES:,}')
    if final >= TARGET_IMAGENES:
        print('  🎉 COMPLETA — lista para entrenamiento')
    else:
        print(f'  ⚠️  Faltan {TARGET_IMAGENES-final:,} — re-ejecuta')
    print('=' * 55)


print('✅ Pipeline de augmentación vía API listo')

In [ ]:
# ════ OPCIONES DEL MENÚ ═══════════════════════════════════════

CARPETA_EXCLUIDA = 'otra'   # se omite al contar "todos los datasets"


def opcion_contar():
    """[1] Contabilizar imágenes .jpg de una carpeta, o de TODOS los
    datasets sumados (excepto «otra»)."""
    nombre, indice, sueltas = construir_indice(FOLDER_ID, incluir_raiz=True)

    # opción especial al final: todos los datasets menos «otra»
    TODOS = max(indice) + 1
    indice[TODOS] = {'name': f'(TODOS los datasets EXCEPTO «{CARPETA_EXCLUIDA}»)',
                     'id': None}

    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    if sueltas:
        print(f'⚠️  {sueltas:,} .jpg sueltos en la raíz '
              f'(la opción [0] los incluye).')

    sel = pedir_opcion(indice, '¿Qué carpeta quieres contabilizar?')

    # ── TODOS los datasets excepto «otra» (conteo PARALELO) ──
    if sel == TODOS:
        datasets = [c for k, c in indice.items()
                    if k not in (0, TODOS)
                    and c['name'].lower() != CARPETA_EXCLUIDA.lower()]
        if not datasets:
            print(f'\n❌ No hay datasets que contar (¿solo existe «{CARPETA_EXCLUIDA}»?).')
            return

        print(f'\n⏳ Contando {len(datasets)} datasets en paralelo '
              f'({NUM_COUNT_WORKERS} hilos, excepto «{CARPETA_EXCLUIDA}»)…')
        _svc = _servicio_por_hilo()

        def _contar_dataset(c):
            return c['name'], _contar_recursivo(c['id'], _svc())

        resultados = []
        with ThreadPoolExecutor(max_workers=NUM_COUNT_WORKERS) as ex:
            futs = {ex.submit(_contar_dataset, c): c for c in datasets}
            with tqdm(total=len(datasets), desc='Datasets', unit=' set',
                      dynamic_ncols=True) as pbar:
                for fut in as_completed(futs):
                    resultados.append(fut.result())
                    pbar.update(1)

        total = sum(n for _, n in resultados)
        print('\n' + '═' * 50)
        print(f'  CONTEO POR DATASET (excepto «{CARPETA_EXCLUIDA}»)')
        print('═' * 50)
        for nom, n in sorted(resultados, key=lambda r: r[1], reverse=True):
            faltan = max(0, TARGET_IMAGENES - n)
            marca  = '✅' if faltan == 0 else f'faltan {faltan:,}'
            print(f'  {nom:<28} {n:>9,}  {marca}')
        print('─' * 50)
        print(f'  {"TOTAL":<28} {total:>9,}')
        print('═' * 50)
        return

    # ── Una sola carpeta (comportamiento original) ───────────
    elegida = indice[sel]
    print(f'\n⏳ Contando .jpg en: {elegida["name"]}…')
    n = contar_imagenes(elegida['id'])

    faltan = max(0, TARGET_IMAGENES - n)
    pct    = n / TARGET_IMAGENES * 100

    print('\n' + '═' * 50)
    print(f'  Carpeta  : {elegida["name"]}')
    print(f'  .jpg     : {n:,}  ({pct:.1f}% de {TARGET_IMAGENES:,})')
    if faltan:
        print(f'  Faltan   : {faltan:,}')
    else:
        print(f'  ✅ Meta alcanzada ({TARGET_IMAGENES:,})')
    print('═' * 50)


def opcion_contar_subcarpetas():
    """[2] Contabilizar una carpeta «por dentro» (vía Drive API).
    Útil cuando una clase vive dentro de otra carpeta (ej. «noise» dentro
    de «otra»). Eliges la carpeta contenedora y decides:
       · DESGLOSAR  → cuenta cada subcarpeta por separado + total
       · TOTAL      → cuenta todo lo de adentro como un solo número
    Todo es API (service.files().list): ve las subcarpetas aunque Drive
    todavía no las muestre en la web."""
    # 1) elegir la carpeta contenedora
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    sel = pedir_opcion(indice, '¿Qué carpeta quieres contabilizar por dentro?')
    elegida = indice[sel]

    # 2) listar subcarpetas directas (vía API)
    subs, sueltas = listar_contenido(elegida['id'])
    if not subs:
        print(f'\n«{elegida["name"]}» no tiene subcarpetas. Cuento su total…')
        n = contar_imagenes(elegida['id'], desc=elegida['name'])
        print('\n' + '═' * 50)
        print(f'  {elegida["name"]:<28} {n:>9,}')
        print('═' * 50)
        return

    # 3) preguntar: desglosar por subcarpeta o total
    print(f'\n«{elegida["name"]}» tiene {len(subs)} subcarpeta(s)'
          + (f' + {sueltas:,} .jpg sueltos' if sueltas else '') + ':')
    for c in subs:
        print(f'   · {c["name"]}')
    desglosar = input(
        '\n¿Desglosar por subcarpeta? '
        '(si = una por una · no = total de todo) (si/no): '
    ).strip().lower() in ('si', 's', 'sí')

    # 4a) TOTAL — todo lo de adentro como un solo número
    if not desglosar:
        print(f'\n⏳ Contando TODO dentro de «{elegida["name"]}» vía API…')
        n = contar_imagenes(elegida['id'], desc=elegida['name'])
        print('\n' + '═' * 50)
        print(f'  Carpeta : {elegida["name"]}')
        print(f'  .jpg    : {n:,}')
        print('═' * 50)
        return

    # 4b) DESGLOSE — cada subcarpeta por separado (progreso por carpeta)
    print(f'\n⏳ Contando {len(subs)} subcarpetas de «{elegida["name"]}» vía API…')
    resultados = []
    for c in subs:
        n = contar_imagenes(c['id'], desc=c['name'])
        resultados.append((c['name'], n))
    if sueltas:
        resultados.append(('(sueltos en la raíz)', sueltas))

    total = sum(n for _, n in resultados)
    print('\n' + '═' * 50)
    print(f'  DESGLOSE DE «{elegida["name"]}»')
    print('═' * 50)
    for nom, n in sorted(resultados, key=lambda r: r[1], reverse=True):
        print(f'  {nom:<28} {n:>9,}')
    print('─' * 50)
    print(f'  {"TOTAL":<28} {total:>9,}')
    print('═' * 50)


def copiar_contenido_recursivo(origen_id, destino_id):
    """Recolecta TODOS los .jpg del árbol bajo origen_id (a cualquier
    profundidad) y los copia APLANADOS dentro de destino_id. El original
    se queda. Salta nombres ya presentes en destino (re-run safe).
    Devuelve (copiados, saltados)."""
    import time

    # 1) recolectar .jpg de TODO el árbol (no solo el nivel directo)
    print('  Recolectando .jpg de todo el árbol…')
    archivos, stack = [], [origen_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif _es_imagen(f):
                    archivos.append(f)
            token = resp.get('nextPageToken')
            if not token:
                break
    print(f'  {len(archivos):,} .jpg encontrados en el árbol')

    # 2) nombres ya presentes en destino (para no re-copiar en re-runs)
    print('  Revisando qué ya existe en destino…')
    existentes, token = set(), None
    while True:
        resp = service.files().list(
            q=f"'{destino_id}' in parents and trashed=false",
            fields='nextPageToken, files(name)',
            pageToken=token, **_ARGS).execute()
        existentes.update(f['name'] for f in resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break

    copiados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    saltados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    to_copy = [f for f in archivos if f['name'] not in existentes]
    saltados['imagenes'] = len(archivos) - len(to_copy)

    # 3) copiar en paralelo — un service por hilo (httplib2 no es thread-safe)
    _svc = _servicio_por_hilo()

    def _copiar_uno(f):
        for intento in range(4):
            try:
                _svc().files().copy(
                    fileId=f['id'],
                    body={'parents': [destino_id], 'name': f['name']},
                    fields='id', supportsAllDrives=True).execute()
                return True, f, None
            except Exception as e:
                err = str(e)
                if '429' in err or 'Rate Limit' in err or 'userRateLimitExceeded' in err:
                    time.sleep(2 ** intento)
                else:
                    return False, f, err
        return False, f, 'Rate limit tras 4 intentos'

    print(f'  {len(to_copy):,} a copiar · {saltados["imagenes"]:,} saltados · {NUM_COPY_WORKERS} hilos')
    with ThreadPoolExecutor(max_workers=NUM_COPY_WORKERS) as executor:
        with tqdm(total=len(to_copy), desc='Copiando', unit=' archivo',
                  dynamic_ncols=True) as pbar:
            for ok, f, err in executor.map(_copiar_uno, to_copy):
                if ok:
                    copiados['imagenes'] += 1
                else:
                    print(f'   ⚠️ No se pudo copiar {f["name"]}: {err}')
                pbar.update(1)

    return copiados, saltados


def opcion_aplanar():
    """[3] Copiar a la carpeta PADRE («la padre de todos», ej. «hora de
    aventura») el contenido de una subcarpeta mal anidada. Deja el original
    (crea duplicados). Si la subcarpeta tiene a su vez más subcarpetas, abre
    un submenú numerado: [0] TODO (sueltos + subcarpetas), cada subcarpeta
    por su número (copia solo esa), o la última = TODAS. Lo elegido se
    recolecta aplanado dentro de la padre."""
    # 1) elegir la carpeta PADRE donde debe caer todo
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    sel = pedir_opcion(indice, '¿En qué carpeta PADRE debe caer todo?')
    destino = indice[sel]

    # 2) buscar subcarpetas dentro de la carpeta padre
    subs, _ = listar_contenido(destino['id'])
    if not subs:
        print(f'\n✅ «{destino["name"]}» no tiene subcarpetas. Nada que copiar.')
        return

    subindice = {i: c for i, c in enumerate(subs, 1)}
    mostrar_indice(subindice,
                   titulo=f'SUBCARPETAS DENTRO DE «{destino["name"]}»')
    sub_sel = pedir_opcion(
        subindice, '¿De qué subcarpeta quieres copiar el contenido?')
    subcarpeta = subindice[sub_sel]

    # 3) ¿la subcarpeta tiene a su vez más subcarpetas? → submenú numerado
    #    fuentes = None → copia directa (sin recursión).
    #    fuentes = lista de (nombre, id) → recolectar recursivo y aplanar.
    nietas, _ = listar_contenido(subcarpeta['id'])
    fuentes = None
    if nietas:
        sub2 = {0: {'name': f'(TODO «{subcarpeta["name"]}» — sueltos + subcarpetas)',
                    'id': subcarpeta['id']}}
        for i, c in enumerate(nietas, 1):
            sub2[i] = c
        TODAS = max(sub2) + 1
        sub2[TODAS] = {'name': '(TODAS las subcarpetas de adentro)', 'id': None}
        mostrar_indice(sub2, titulo=f'¿QUÉ COPIAR DE «{subcarpeta["name"]}»?')
        n_sel = pedir_opcion(sub2, '¿Qué quieres copiar a la padre?')
        if n_sel == TODAS:
            fuentes = [(c['name'], c['id']) for c in nietas]
        else:
            fuentes = [(sub2[n_sel]['name'], sub2[n_sel]['id'])]

    # 4) confirmar (operación que modifica tu Drive)
    if fuentes is None:
        alcance = f'el nivel directo de «{subcarpeta["name"]}»'
    elif len(fuentes) == 1:
        alcance = f'TODO «{fuentes[0][0]}» (recursivo, aplanado)'
    else:
        alcance = f'TODO de {len(fuentes)} subcarpetas (recursivo, aplanado)'
    print(f'\nSe COPIARÁ {alcance}')
    print(f'   →   «{destino["name"]}»')
    print('   (el original se conserva · crea duplicados)')
    if input('¿Confirmas? (si/no): ').strip().lower() not in ('si', 's', 'sí'):
        print('Cancelado. No se copió nada.')
        return

    # 5) copiar
    print('\n⏳ Copiando…')
    if fuentes is None:
        copiados, saltados = copiar_contenido(subcarpeta['id'], destino['id'])
    else:
        copiados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
        saltados = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
        for nom, fid in fuentes:
            if len(fuentes) > 1:
                print(f'\n── {nom} ──')
            c, s = copiar_contenido_recursivo(fid, destino['id'])
            for k in copiados:
                copiados[k] += c[k]
                saltados[k] += s[k]

    total = sum(copiados.values())
    print('\n' + '═' * 50)
    print(f'  Copiados a «{destino["name"]}» : {total:,} elementos')
    print(f'    · .jpg     : {copiados["imagenes"]:,}')
    if copiados['carpetas']:
        print(f'    · carpetas : {copiados["carpetas"]:,}')
    if copiados['otros']:
        print(f'    · otros    : {copiados["otros"]:,}')
    ya = saltados['imagenes'] + saltados['otros']
    if ya:
        print(f'  Saltados (ya existían en destino) : {ya:,}')
    if saltados['carpetas']:
        print(f'  Subcarpetas omitidas (copy no copia carpetas) : {saltados["carpetas"]:,}')
    print('  El contenido original sigue en la subcarpeta.')
    print('═' * 50)


def opcion_augmentar():
    """[4] Completar a 100k: cuenta primero y augmenta SOLO el déficit.
    Usa la ruta MONTADA (acceso directo en MyDrive), igual que el
    notebook de augmentación original."""
    # 1) montar Drive (necesario para leer/escribir vía el acceso directo)
    from google.colab import drive
    drive.mount('/content/drive')

    root = Path(MOUNT_ROOT)
    if not root.is_dir():
        print(f'\n❌ No existe la ruta montada: {root}')
        print('   Crea un acceso directo de la carpeta compartida en tu "Mi unidad"')
        print('   (clic derecho → Organizar → Añadir acceso directo → Mi unidad)')
        print(f'   y verifica NOMBRE_ACCESO_DIRECTO en la config (cell-2).')
        return

    # 2) índice de clases (subcarpetas de la ruta) — mismo UX que [3]
    clases = sorted([d for d in root.iterdir() if d.is_dir()],
                    key=lambda d: d.name.lower())
    if not clases:
        print(f'\n❌ No hay subcarpetas dentro de {root}')
        return

    indice = {i: d for i, d in enumerate(clases, 1)}
    print(f'\n📂 Acceso directo : {root}')
    print('═' * 50)
    print(f'CLASES EN «{root.name}»')
    print('═' * 50)
    for num, d in indice.items():
        print(f'  [{num:>2}]  {d.name}')
    print('═' * 50)

    while True:
        sel = input('\n¿Qué carpeta quieres completar a 100k? (número): ').strip()
        if sel.isdigit() and int(sel) in indice:
            break
        print('   ⚠️  Número inválido, intenta de nuevo.')
    clase_dir = indice[int(sel)]

    # 3) contar + augmentar el déficit (pipeline tal cual)
    print(f'\n⏳ Procesando «{clase_dir.name}»…')
    augmentar_clase(clase_dir.name, clase_dir)


def opcion_augmentar_api():
    """[5] Completar a 100k vía Drive API (sin FUSE).
    Para carpetas tan grandes que la opción [4] falla con
    Input/output error. No necesita el acceso directo en MyDrive:
    usa FOLDER_ID directo, igual que [1] y [3]."""
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=False)
    if not indice:
        print('\n❌ No hay subcarpetas (clases) en la carpeta raíz.')
        return
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice, titulo='CLASES (subcarpetas)')
    sel = pedir_opcion(indice, '¿Qué clase quieres completar a 100k?')
    clase = indice[sel]
    print(f'\n⏳ Procesando «{clase["name"]}» vía API…')
    augmentar_clase_api(clase['id'], clase['name'])


print('✅ Opciones del menú listas')

In [ ]:
# ════ PIPELINE: EXTRAER COMPRIMIDOS (.rar / .zip) — opción [6] ═
# Descarga un comprimido de Drive vía API, lo descomprime en local (Colab)
# y sube sus imágenes "tal cual" (mismo nombre y tamaño) a la carpeta PADRE.
# NO estandariza ni renombra (para eso está estandarizar_local.py).
# Re-run safe: salta nombres ya presentes en destino.
import io as _io2
import os as _os2
import time
import shutil as _shutil2
import subprocess as _sub2
import zipfile as _zip2
import mimetypes as _mime2
from googleapiclient.http import MediaFileUpload

EXT_COMPRIMIDOS = {'.rar', '.zip'}
# Imágenes que se suben tal cual. Más amplio que el .jpg del resto del
# notebook: un comprimido puede traer png/webp y no queremos perderlas.
EXT_IMG_EXTRAER = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}
TMP_EXTRAER     = '/content/tmp_extraer'


def _es_comprimido(f):
    return Path(f['name']).suffix.lower() in EXT_COMPRIMIDOS


def _buscar_comprimidos(folder_id):
    """Recorre TODO el árbol bajo folder_id y devuelve los .rar/.zip."""
    encontrados, stack = [], [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif _es_comprimido(f):
                    encontrados.append(f)
            token = resp.get('nextPageToken')
            if not token:
                break
    return encontrados


def _asegurar_unrar():
    """Instala unrar la primera vez (Colab corre apt como root)."""
    if _shutil2.which('unrar'):
        return True
    print('  Instalando unrar…')
    _sub2.run(['apt-get', '-qq', 'install', '-y', 'unrar'], capture_output=True)
    return _shutil2.which('unrar') is not None


def _extraer_a_local(data, nombre, out_dir):
    """Descomprime los bytes de un .rar/.zip en out_dir y devuelve las rutas
    de las imágenes extraídas (recursivo, a cualquier profundidad)."""
    ext = Path(nombre).suffix.lower()
    _os2.makedirs(out_dir, exist_ok=True)
    if ext == '.zip':
        try:
            with _zip2.ZipFile(_io2.BytesIO(data)) as z:
                z.extractall(out_dir)
        except Exception as e:
            print(f'   ⚠️ zip inválido «{nombre}»: {str(e)[:160]}')
            return []
    elif ext == '.rar':
        if not _asegurar_unrar():
            print('   ⚠️ No se pudo instalar unrar — se omite este .rar')
            return []
        rar_path = _os2.path.join(out_dir, '_tmp.rar')
        with open(rar_path, 'wb') as fh:
            fh.write(data)
        r = _sub2.run(['unrar', 'x', '-o+', '-idq', rar_path, out_dir + '/'],
                      capture_output=True, text=True)
        _os2.remove(rar_path)
        if r.returncode != 0:
            print(f'   ⚠️ unrar falló en «{nombre}»: {r.stderr.strip()[:200]}')
            return []
    # recolectar imágenes de todo el árbol extraído (se aplanan por basename)
    imgs = []
    for raiz, _, files in _os2.walk(out_dir):
        for fn in files:
            if Path(fn).suffix.lower() in EXT_IMG_EXTRAER:
                imgs.append(_os2.path.join(raiz, fn))
    return imgs


def opcion_extraer_comprimidos():
    """[6] Extraer un .rar/.zip que vive dentro de una subcarpeta y subir sus
    imágenes TAL CUAL (mismo nombre y tamaño) a la carpeta PADRE. No
    estandariza ni renombra. Salta nombres ya presentes en destino."""
    # 1) elegir la carpeta PADRE (destino)
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    sel = pedir_opcion(indice, '¿En qué carpeta PADRE deben caer las imágenes?')
    destino = indice[sel]

    # 2) elegir DÓNDE está el comprimido (la propia padre o una subcarpeta)
    subs, _ = listar_contenido(destino['id'])
    if not subs:
        print(f'\n«{destino["name"]}» no tiene subcarpetas; busco en ella misma.')
        origen = destino
    else:
        subindice = {0: {'name': f'(la propia «{destino["name"]}»)',
                         'id': destino['id']}}
        for i, c in enumerate(subs, 1):
            subindice[i] = c
        mostrar_indice(
            subindice,
            titulo=f'¿DÓNDE ESTÁ EL .rar/.zip? (dentro de «{destino["name"]}»)')
        origen = subindice[pedir_opcion(
            subindice, '¿En qué carpeta está el comprimido?')]

    # 3) buscar comprimidos en el árbol del origen
    print(f'\n🔎 Buscando .rar/.zip dentro de «{origen["name"]}»…')
    comprimidos = _buscar_comprimidos(origen['id'])
    if not comprimidos:
        print('   No se encontró ningún .rar/.zip. Nada que extraer.')
        return
    print(f'   {len(comprimidos)} comprimido(s):')
    for f in comprimidos:
        print(f'      · {f["name"]}')

    # 4) confirmar (sube archivos a tu Drive)
    print('\nSe DESCARGARÁ, descomprimirá y subirá su contenido a:')
    print(f'   →   «{destino["name"]}»  (tal cual, sin estandarizar)')
    if input('¿Confirmas? (si/no): ').strip().lower() not in ('si', 's', 'sí'):
        print('Cancelado. No se subió nada.')
        return

    # 5) nombres ya presentes en destino (para no re-subir en re-runs)
    print('  Revisando qué ya existe en destino…')
    existentes, token = set(), None
    while True:
        resp = service.files().list(
            q=f"'{destino['id']}' in parents and trashed=false",
            fields='nextPageToken, files(name)',
            pageToken=token, **_ARGS).execute()
        existentes.update(f['name'] for f in resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break

    # 6) por cada comprimido: descargar → extraer → subir lo nuevo (paralelo)
    _shutil2.rmtree(TMP_EXTRAER, ignore_errors=True)
    _svc = _servicio_por_hilo()
    subidas = saltadas = fallidas = 0

    def _subir_uno(args):
        ruta, name = args
        mt = _mime2.guess_type(name)[0] or 'application/octet-stream'
        for intento in range(4):
            try:
                _svc().files().create(
                    body={'name': name, 'parents': [destino['id']]},
                    media_body=MediaFileUpload(ruta, mimetype=mt, resumable=False),
                    fields='id', supportsAllDrives=True).execute()
                return True, name
            except Exception as e:
                err = str(e)
                if '429' in err or 'Rate Limit' in err or 'userRateLimitExceeded' in err:
                    time.sleep(2 ** intento)
                else:
                    print(f'   ⚠️ No se pudo subir {name}: {err[:120]}')
                    return False, name
        return False, name

    for f in comprimidos:
        print(f'\n── {f["name"]} ──')
        print('  Descargando…')
        try:
            data = service.files().get_media(
                fileId=f['id'], supportsAllDrives=True).execute()
        except Exception as e:
            print(f'   ⚠️ No se pudo descargar: {str(e)[:160]}')
            fallidas += 1
            continue
        out_dir = _os2.path.join(TMP_EXTRAER, Path(f['name']).stem)
        imgs = _extraer_a_local(data, f['name'], out_dir)
        print(f'  {len(imgs):,} imagen(es) dentro del comprimido')

        # aplanar por basename; saltar las que ya existen (en destino o ya vistas)
        tareas, vistos = [], set()
        for ruta in imgs:
            name = Path(ruta).name
            if name in existentes or name in vistos:
                saltadas += 1
                continue
            vistos.add(name)
            tareas.append((ruta, name))

        if tareas:
            with ThreadPoolExecutor(max_workers=NUM_COPY_WORKERS) as ex:
                with tqdm(total=len(tareas), desc='Subiendo', unit=' img',
                          dynamic_ncols=True) as pbar:
                    for ok, name in ex.map(_subir_uno, tareas):
                        if ok:
                            subidas += 1
                            existentes.add(name)   # evita duplicar entre comprimidos
                        else:
                            fallidas += 1
                        pbar.update(1)
        _shutil2.rmtree(out_dir, ignore_errors=True)

    _shutil2.rmtree(TMP_EXTRAER, ignore_errors=True)
    print('\n' + '═' * 50)
    print(f'  Subidas a «{destino["name"]}» : {subidas:,}')
    if saltadas:
        print(f'  Saltadas (ya existían / duplicadas) : {saltadas:,}')
    if fallidas:
        print(f'  Fallidas : {fallidas:,}')
    print('  Las imágenes se subieron TAL CUAL (sin estandarizar).')
    print('═' * 50)


print('✅ Pipeline de extracción de comprimidos listo')

## ▶️ Menú — ejecuta esta celda

In [ ]:
# ════ MENÚ PRINCIPAL ══════════════════════════════════════════
print('═' * 50)
print('  MENÚ PRINCIPAL')
print('═' * 50)
print('  [1]  Contabilizar imágenes de una carpeta')
print('  [2]  Contabilizar una carpeta por dentro (desglose por subcarpeta · API)')
print('  [3]  Copiar una subcarpeta a la carpeta PADRE (recursivo opcional · deja original)')
print('  [4]  Completar a 100k: augmentar el déficit (vía mount/FUSE)')
print('  [5]  Completar a 100k vía API (carpetas grandes, sin FUSE)')
print('  [6]  Extraer .rar/.zip de una subcarpeta → subir imágenes a la PADRE (tal cual)')
print('═' * 50)

_op = input('Elige una opción (1/2/3/4/5/6): ').strip()
if _op == '1':
    opcion_contar()
elif _op == '2':
    opcion_contar_subcarpetas()
elif _op == '3':
    opcion_aplanar()
elif _op == '4':
    opcion_augmentar()
elif _op == '5':
    opcion_augmentar_api()
elif _op == '6':
    opcion_extraer_comprimidos()
else:
    print('Opción inválida. Vuelve a ejecutar esta celda.')